# Statistical Significance Testing

## 1. Objective

Determine whether the predictive-performance differences between Naive, Persistence-Enhanced LSTM, and Chronos-Bolt-Tiny are statistically significant on the same Bitcoin daily test period.

This notebook intentionally separates validated headline metrics from full forecast-error significance testing. Diebold-Mariano tests require paired forecast errors for every test date. If full validated forecast vectors are unavailable, the notebook stops rather than inventing statistical results from aggregate metrics.


In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_bitcoin_data
from src.preprocessing import prepare_daily_bitcoin_data
from src.metrics import mae, rmse, mape, smape

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.6f}".format)


## 2. Load Forecast Results

The Bitcoin test target and Naive forecast are reconstructed directly from the raw dataset because they are deterministic and do not require model fitting.

Persistence-Enhanced LSTM and Chronos-Bolt-Tiny must come from validated forecast vectors. Aggregate metrics alone are insufficient for significance testing. The notebook looks for a CSV with columns:

- `Date`
- `y_test`
- `naive_forecast`
- `persistence_enhanced_lstm`
- `chronos_bolt_tiny_forecast`

Searched locations:

- `artifacts/validated_forecasts.csv`
- `outputs/validated_forecasts.csv`
- `data/processed/validated_forecasts.csv`
- `data/validated_forecasts.csv`

If the file is missing, export the validated forecast vectors from the executed model notebooks first. Do not substitute synthetic forecasts.


In [ ]:
data_path = PROJECT_ROOT / "data" / "bitcoin" / "btcusd_1-min_data.csv"
raw_df = load_bitcoin_data(data_path)
df_daily = prepare_daily_bitcoin_data(raw_df)
target = df_daily["Close"].asfreq("D").dropna().rename("Close")

split_idx = int(len(target) * 0.8)
train = target.iloc[:split_idx]
test = target.iloc[split_idx:]
y_test = test.rename("y_test")
naive_forecast = target.shift(1).reindex(test.index).rename("naive_forecast")

split_verification = pd.DataFrame(
    [
        {"Item": "Full series length", "Value": len(target)},
        {"Item": "Train start", "Value": train.index.min()},
        {"Item": "Train end", "Value": train.index.max()},
        {"Item": "Test start", "Value": test.index.min()},
        {"Item": "Test end", "Value": test.index.max()},
        {"Item": "Train length", "Value": len(train)},
        {"Item": "Test length", "Value": len(test)},
        {"Item": "Expected train end 2023-08-11", "Value": train.index.max() == pd.Timestamp("2023-08-11", tz="UTC")},
        {"Item": "Expected test start 2023-08-12", "Value": test.index.min() == pd.Timestamp("2023-08-12", tz="UTC")},
    ]
)
split_verification


In [ ]:
forecast_file_candidates = [
    PROJECT_ROOT / "artifacts" / "validated_forecasts.csv",
    PROJECT_ROOT / "outputs" / "validated_forecasts.csv",
    PROJECT_ROOT / "data" / "processed" / "validated_forecasts.csv",
    PROJECT_ROOT / "data" / "validated_forecasts.csv",
]

forecast_file = next((path for path in forecast_file_candidates if path.exists()), None)

if forecast_file is None:
    forecast_results = pd.DataFrame(index=test.index)
    forecast_results["y_test"] = y_test
    forecast_results["naive_forecast"] = naive_forecast
    forecast_results["persistence_enhanced_lstm"] = np.nan
    forecast_results["chronos_bolt_tiny_forecast"] = np.nan
    forecast_source_status = pd.DataFrame(
        [
            {"Forecast": "y_test", "Source": "Reconstructed from Bitcoin Close series", "Available": True},
            {"Forecast": "naive_forecast", "Source": "target.shift(1).reindex(test.index)", "Available": True},
            {"Forecast": "persistence_enhanced_lstm", "Source": "Validated forecast vector artifact required", "Available": False},
            {"Forecast": "chronos_bolt_tiny_forecast", "Source": "Validated forecast vector artifact required", "Available": False},
        ]
    )
else:
    forecast_results = pd.read_csv(forecast_file, parse_dates=["Date"]).set_index("Date")
    if forecast_results.index.tz is None:
        forecast_results.index = forecast_results.index.tz_localize("UTC")
    else:
        forecast_results.index = forecast_results.index.tz_convert("UTC")
    forecast_results = forecast_results.reindex(test.index)
    forecast_source_status = pd.DataFrame(
        [
            {"Forecast": "all forecasts", "Source": str(forecast_file), "Available": True},
        ]
    )

forecast_source_status


In [ ]:
required_columns = [
    "y_test",
    "naive_forecast",
    "persistence_enhanced_lstm",
    "chronos_bolt_tiny_forecast",
]

missing_columns = [column for column in required_columns if column not in forecast_results.columns]
if missing_columns:
    raise ValueError(f"Missing required forecast columns: {missing_columns}")

forecast_results["y_test"] = forecast_results["y_test"].fillna(y_test)
forecast_results["naive_forecast"] = forecast_results["naive_forecast"].fillna(naive_forecast)

persistence_enhanced_lstm = forecast_results["persistence_enhanced_lstm"].rename("Persistence-Enhanced LSTM")
chronos_bolt_tiny_forecast = forecast_results["chronos_bolt_tiny_forecast"].rename("Chronos-Bolt-Tiny")

availability_table = pd.DataFrame(
    [
        {
            "Series": column,
            "Length": len(forecast_results[column]),
            "Missing Values": int(forecast_results[column].isna().sum()),
            "Index Equals Test Index": forecast_results[column].index.equals(test.index),
        }
        for column in required_columns
    ]
)
availability_table


In [ ]:
if forecast_results[["persistence_enhanced_lstm", "chronos_bolt_tiny_forecast"]].isna().any().any():
    raise FileNotFoundError(
        "Full validated forecast vectors for Persistence-Enhanced LSTM and Chronos-Bolt-Tiny are required "
        "for statistical significance testing. Aggregate metrics are not enough for Diebold-Mariano tests. "
        "Create one of the documented validated_forecasts.csv artifacts, then rerun this notebook."
    )


## 3. Forecast Error Construction

In [ ]:
assert len(y_test) == len(naive_forecast) == len(persistence_enhanced_lstm) == len(chronos_bolt_tiny_forecast)
assert y_test.index.equals(naive_forecast.index)
assert y_test.index.equals(persistence_enhanced_lstm.index)
assert y_test.index.equals(chronos_bolt_tiny_forecast.index)
assert not pd.concat([y_test, naive_forecast, persistence_enhanced_lstm, chronos_bolt_tiny_forecast], axis=1).isna().any().any()

naive_errors = y_test - naive_forecast
lstm_errors = y_test - persistence_enhanced_lstm
chronos_errors = y_test - chronos_bolt_tiny_forecast

error_frame = pd.DataFrame(
    {
        "Naive": naive_errors,
        "Persistence-Enhanced LSTM": lstm_errors,
        "Chronos-Bolt-Tiny": chronos_errors,
    }
)

forecast_validation = pd.DataFrame(
    [
        {"Check": "Identical forecast lengths", "Result": len(set([len(y_test), len(naive_forecast), len(persistence_enhanced_lstm), len(chronos_bolt_tiny_forecast)])) == 1},
        {"Check": "Naive index equals y_test", "Result": naive_forecast.index.equals(y_test.index)},
        {"Check": "LSTM index equals y_test", "Result": persistence_enhanced_lstm.index.equals(y_test.index)},
        {"Check": "Chronos index equals y_test", "Result": chronos_bolt_tiny_forecast.index.equals(y_test.index)},
        {"Check": "No missing values", "Result": not error_frame.isna().any().any()},
    ]
)
forecast_validation


## 4. Diebold–Mariano Test

In [ ]:
def diebold_mariano_test(actual, forecast1, forecast2, power=2):
    actual = pd.Series(actual).astype(float)
    forecast1 = pd.Series(forecast1).astype(float)
    forecast2 = pd.Series(forecast2).astype(float)
    aligned = pd.concat(
        [
            actual.rename("actual"),
            forecast1.rename("forecast1"),
            forecast2.rename("forecast2"),
        ],
        axis=1,
    ).dropna()

    if aligned.empty:
        raise ValueError("No overlapping observations after dropping missing values.")

    loss1 = np.abs(aligned["actual"] - aligned["forecast1"]) ** power
    loss2 = np.abs(aligned["actual"] - aligned["forecast2"]) ** power
    loss_differential = loss1 - loss2
    sample_size = len(loss_differential)

    if sample_size < 2:
        raise ValueError("At least two observations are required for the Diebold-Mariano test.")

    mean_loss_differential = loss_differential.mean()
    variance = loss_differential.var(ddof=1)

    if np.isclose(variance, 0):
        dm_statistic = np.inf if mean_loss_differential > 0 else -np.inf if mean_loss_differential < 0 else 0.0
        p_value = 0.0 if not np.isclose(mean_loss_differential, 0) else 1.0
    else:
        dm_statistic = mean_loss_differential / np.sqrt(variance / sample_size)
        p_value = 2 * stats.t.sf(np.abs(dm_statistic), df=sample_size - 1)

    return {
        "DM Statistic": dm_statistic,
        "p-value": p_value,
        "Mean Loss Differential": mean_loss_differential,
        "Sample Size": sample_size,
    }


## 5. Pairwise Model Comparisons

In [ ]:
forecasts_for_testing = {
    "Naive": naive_forecast,
    "Persistence-Enhanced LSTM": persistence_enhanced_lstm,
    "Chronos-Bolt-Tiny": chronos_bolt_tiny_forecast,
}

pairwise_comparisons = [
    ("Naive", "Persistence-Enhanced LSTM"),
    ("Naive", "Chronos-Bolt-Tiny"),
    ("Persistence-Enhanced LSTM", "Chronos-Bolt-Tiny"),
]

dm_rows = []
for model_a, model_b in pairwise_comparisons:
    result = diebold_mariano_test(y_test, forecasts_for_testing[model_a], forecasts_for_testing[model_b], power=2)
    metrics_a = {
        "MAE": mae(y_test, forecasts_for_testing[model_a]),
        "RMSE": rmse(y_test, forecasts_for_testing[model_a]),
    }
    metrics_b = {
        "MAE": mae(y_test, forecasts_for_testing[model_b]),
        "RMSE": rmse(y_test, forecasts_for_testing[model_b]),
    }
    winner = model_a if metrics_a["RMSE"] < metrics_b["RMSE"] else model_b
    dm_rows.append(
        {
            "Comparison": f"{model_a} vs {model_b}",
            **result,
            "Mean Error Difference": (y_test - forecasts_for_testing[model_a]).mean() - (y_test - forecasts_for_testing[model_b]).mean(),
            "Winner": winner,
            "Significant at alpha=0.05": result["p-value"] < 0.05,
            "Interpretation Rule": "Statistically significant difference detected." if result["p-value"] < 0.05 else "No statistically significant difference in predictive accuracy.",
        }
    )

dm_results = pd.DataFrame(dm_rows).set_index("Comparison")
dm_results


## 6. Error Distribution Analysis

In [ ]:
error_summary = error_frame.agg(["mean", "std", "median", "min", "max"]).T
error_summary["MAE"] = error_frame.abs().mean()
error_summary["RMSE"] = np.sqrt((error_frame ** 2).mean())
error_summary


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, column in zip(axes, error_frame.columns):
    error_frame[column].plot(kind="hist", bins=50, density=True, alpha=0.55, ax=ax, label="Histogram")
    try:
        error_frame[column].plot(kind="kde", ax=ax, label="KDE")
    except Exception:
        pass
    ax.set_title(f"{column} Errors")
    ax.set_xlabel("Forecast Error")
    ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
error_frame.plot(kind="box", ax=ax)
ax.set_title("Forecast Error Boxplots")
ax.set_ylabel("Forecast Error")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, column in zip(axes, error_frame.columns):
    stats.probplot(error_frame[column].dropna(), dist="norm", plot=ax)
    ax.set_title(f"Q-Q Plot: {column}")
plt.tight_layout()
plt.show()


## 7. Effect Size Analysis

In [ ]:
def cohens_d_absolute_errors(actual, forecast1, forecast2):
    abs_error_1 = (pd.Series(actual) - pd.Series(forecast1)).abs()
    abs_error_2 = (pd.Series(actual) - pd.Series(forecast2)).abs()
    aligned = pd.concat([abs_error_1.rename("a"), abs_error_2.rename("b")], axis=1).dropna()
    difference = aligned["a"] - aligned["b"]
    if np.isclose(difference.std(ddof=1), 0):
        return np.nan
    return difference.mean() / difference.std(ddof=1)


effect_rows = []
for model_a, model_b in pairwise_comparisons:
    forecast_a = forecasts_for_testing[model_a]
    forecast_b = forecasts_for_testing[model_b]
    mae_a = mae(y_test, forecast_a)
    mae_b = mae(y_test, forecast_b)
    rmse_a = rmse(y_test, forecast_a)
    rmse_b = rmse(y_test, forecast_b)
    effect_rows.append(
        {
            "Comparison": f"{model_a} vs {model_b}",
            "MAE Difference": mae_a - mae_b,
            "RMSE Difference": rmse_a - rmse_b,
            "Percentage Improvement of Model B vs Model A": (mae_a - mae_b) / mae_a * 100,
            "Cohen d on Absolute Errors": cohens_d_absolute_errors(y_test, forecast_a, forecast_b),
        }
    )

effect_size_table = pd.DataFrame(effect_rows).set_index("Comparison")
effect_size_table


## 8. Practical Significance

In [ ]:
average_bitcoin_price = y_test.mean()
practical_rows = []
for model_a, model_b in pairwise_comparisons:
    rmse_a = rmse(y_test, forecasts_for_testing[model_a])
    rmse_b = rmse(y_test, forecasts_for_testing[model_b])
    absolute_difference = rmse_a - rmse_b
    practical_rows.append(
        {
            "Comparison": f"{model_a} vs {model_b}",
            "RMSE Difference": absolute_difference,
            "RMSE Percentage Difference": absolute_difference / rmse_a * 100,
            "Difference Relative to Average Bitcoin Price": absolute_difference / average_bitcoin_price * 100,
            "Practical Importance": "Small" if abs(absolute_difference / average_bitcoin_price * 100) < 1 else "Moderate/Large",
        }
    )

practical_significance = pd.DataFrame(practical_rows).set_index("Comparison")
practical_significance


## 9. Final Interpretation

In [ ]:
final_summary = dm_results.join(effect_size_table, how="left").join(practical_significance, how="left")
final_summary["Significant?"] = final_summary["p-value"] < 0.05
final_summary = final_summary[
    [
        "DM Statistic",
        "p-value",
        "Significant?",
        "Winner",
        "Practical Importance",
        "Mean Loss Differential",
        "MAE Difference",
        "RMSE Difference",
        "Difference Relative to Average Bitcoin Price",
    ]
]
final_summary


### Research Interpretation

Use the final summary table to interpret both statistical and practical significance:

- If `p-value > 0.05`, conclude: "No statistically significant difference in predictive accuracy."
- If `p-value < 0.05`, conclude: "Statistically significant difference detected."
- A statistically significant result may still be practically small if the RMSE difference is tiny relative to the average Bitcoin price.
- Naive should only be described as truly outperforming another model if the paired error comparison and practical-effect analysis support that claim.
- Chronos-Bolt-Tiny and Persistence-Enhanced LSTM should be treated as statistically similar to Naive when p-values exceed `0.05`, even if headline RMSE differs slightly.
- For trustworthy forecasting, statistical significance should be considered alongside protocol comparability, diagnostics, robustness, and failure detectability.


## 10. Key Findings

This notebook is designed to produce no fabricated findings. The key findings should be filled from the executed Diebold-Mariano and effect-size tables after validated forecast vectors are available.

Expected outputs after execution with complete forecast vectors:

- Pairwise Diebold-Mariano table.
- Error summary table.
- Effect-size table.
- Practical-significance table.
- Final summary table with statistical and practical interpretation.
